# 三、数据滚雪球： 基于一构建的数据集扩展数据

### 1.sorted_users_test.csv文件中存储了用户的信息，接下来我想扫描/home/wangshuo/resource/DataSets/parler/csv_data/pc/comments 下31个comments文件，commnets文件中的creator字段和sorted_users_test.csv中的id相对应，这两张表作左连接，只要commnets文件中的creator字段在sorted_users_test 中的id有对应，就保存这个commnet元组，并统计sorted_users_test 发布了多少这个commnet评论

In [ ]:
import os
import pandas as pd
from tqdm import tqdm  # 用于显示进度条

# 设置路径
sorted_users_file = '/home/wangshuo/resource/DataSets/parler/csv_data/sub_data/add_property/sorted_users_test.csv'
comments_dir = '/home/wangshuo/resource/DataSets/parler/csv_data/pc/comments'
output_file = '/home/wangshuo/resource/DataSets/parler/csv_data/sub_data/filtered_comments.csv'

# 读取sorted_users_test.csv
print("读取 sorted_users_test.csv 文件...")
sorted_users_df = pd.read_csv(sorted_users_file)

# 初始化结果数据框
filtered_comments_list = []

# 扫描 comments 目录
comment_files = [f for f in os.listdir(comments_dir) if f.endswith('.csv')]

print("开始处理 comments 文件...")
for comment_file in tqdm(comment_files, desc="扫描 comments 文件"):
    # 读取评论文件
    comment_file_path = os.path.join(comments_dir, comment_file)
    comments_df = pd.read_csv(comment_file_path)
    
    # 只保留 creator 在 sorted_users_df 的 id 中的记录
    filtered_comments = comments_df[comments_df['creator'].isin(sorted_users_df['id'])]
    filtered_comments_list.append(filtered_comments)

# 合并所有过滤后的评论数据
filtered_comments_df = pd.concat(filtered_comments_list, ignore_index=True)

# 统计每个用户的评论数量
comment_counts = filtered_comments_df['creator'].value_counts().reset_index()
comment_counts.columns = ['id', 'comment_count']

# 将评论数量统计结果与 sorted_users_df 左连接
print("统计评论数量并保存结果...")
final_result = pd.merge(sorted_users_df, comment_counts, on='id', how='left')
final_result['comment_count'] = final_result['comment_count'].fillna(0).astype(int)

# 保存过滤后的评论数据
filtered_comments_df.to_csv(output_file, index=False)

# 保存统计结果
final_result.to_csv('/home/wangshuo/resource/DataSets/parler/csv_data/sub_data/middle_daset/comment_counts.csv', index=False)

# 打印统计结果的前几行
print("统计结果示例：")
print(final_result[['id', 'post_sum', 'comment_sum', 'comment_count']].head())


In [ ]:
print(len(final_result))
print(len(filtered_comments_df))
# 设置打印条数
max_rows = 5000

print("统计结果示例：")
for idx, row in final_result[['id', 'post_sum', 'comment_sum', 'comment_count']].head(max_rows).iterrows():
    print(f"ID: {row['id']}, Post Sum: {row['post_sum']}, Comment Sum: {row['comment_sum']}, Comment Count: {row['comment_count']}")


### 2.filtered_comments.csv文件已经顺利保存接下来遍历文件夹下所有posts文件，filtered_comments对于每个posts文件作左连接，
将与filtered_comments中有对应的post元组保存下来，最后将filtered_comments中有对应的post元组集合按commnets字段降序排序，
保存到posts_test_large.csv文件中，不要修改filtered_comments.csv文件的内容,也就是说只要posts文件中的一个post元组在filtered_comments.csv，
能够找到对应的comment，就把这个post元组记录下来，最后保存这些集合到posts_test_large.csv文件中

In [ ]:
# 初始化结果数据框
matched_posts_list = []

# 遍历 posts 目录下的所有文件
post_files = [f for f in os.listdir(posts_dir) if f.endswith('.csv')]

print("开始处理 posts 文件...")
for post_file in tqdm(post_files, desc="扫描 posts 文件"):
    # 读取 posts 文件
    post_file_path = os.path.join(posts_dir, post_file)
    posts_df = pd.read_csv(post_file_path)
    
    # 筛选出 posts 文件中在 filtered_comments 中有对应 post 的记录
    matched_posts = posts_df[posts_df['id'].isin(filtered_comments_df['post'])]
    matched_posts_list.append(matched_posts)

# 合并所有匹配的 posts 数据
final_posts_df = pd.concat(matched_posts_list, ignore_index=True)

# 按 comments 字段降序排序
final_posts_df = final_posts_df.sort_values(by='comments', ascending=False)

# 保存到 posts_test_large.csv
print("保存结果到 posts_test_large.csv...")
final_posts_df.to_csv(output_file, index=False)

print("完成！")


In [ ]:
print(len(final_posts_df))

### 3.posts_test_large.csv成功保存，接下来遍历filtered_comments.csv文件，检查文件每一个元组comment的post属性，
是否在posts_test_large.csv中有对应的post，如果有则保存到文件comments_test_large.csv中，没有则不保存

In [ ]:
import pandas as pd

# 文件路径
date_dir = '/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5/'
filtered_comments_file = date_dir + 'comment.csv'
posts_test_large_file = date_dir + 'post.csv'
output_file = date_dir + 'comment_lack.csv'

# 读取文件
print("读取 comment.csv 文件...")
filtered_comments_df = pd.read_csv(filtered_comments_file)
print("comment.csv 行数:", len(filtered_comments_df))

print("读取 post.csv 文件...")
posts_test_large_df = pd.read_csv(posts_test_large_file)
print("post.csv 行数:", len(posts_test_large_df))

# 筛选
print("筛选匹配的 comments...")
matched_comments_df = filtered_comments_df[filtered_comments_df['post'].isin(posts_test_large_df['id:ID'])]
print("匹配到的 comment 行数:", len(matched_comments_df))

# 保存结果
print("保存结果到 comment_lack.csv...")
matched_comments_df.to_csv(output_file, index=False)

# 再读回检查保存的行数
saved_df = pd.read_csv(output_file)
print("comment_lack.csv 行数:", len(saved_df))

print("完成！")


### 4.读取comments_test_large.csv，posts_test_large.csv,users_test.csv,给出这三个文件分别有多少元组，
comments文件中元组通过外键post和posts文件的id相连接 ，
posts 和comments文件中元组通过creator与users的id连接，
计算每个post都有多少comments： 将结果作为新属性pcNum保存到 posts_test_large 原文件中
计算每个user发布了多少comments： 将结果作为新属性ucNum保存到 users_test 原文件中
计算每个user发布了多少posts： 将结果作为新属性upNum保存到 users_test 原文件中
对于更新后的users_test.csv文件按照ucNum降序排列

In [ ]:
import pandas as pd

# 文件路径
base_path = '/home/wangshuo/resource/datasets/parler/csv_data/sub_data/middle_daset/'
comments_file = base_path + 'sorted_comment.csv'
posts_file = base_path + 'sorted_post.csv'
users_file = base_path + 'sorted_user.csv'

# 读取文件
print("读取文件...")
comments_df = pd.read_csv(comments_file)
posts_df = pd.read_csv(posts_file)
users_df = pd.read_csv(users_file)

# 输出元组数量
comments_count = len(comments_df)
posts_count = len(posts_df)
users_count = len(users_df)

print(f"comments_test_large.csv 有 {comments_count} 个元组。")
print(f"posts_test_large.csv 有 {posts_count} 个元组。")
print(f"users_test.csv 有 {users_count} 个元组。")

# 计算每个 post 的评论数
print("计算每个 post 的评论数...")
post_comment_counts = comments_df.groupby('post').size().reset_index(name='pcNum')
# posts_df = posts_df.merge(post_comment_counts, how='left', left_on='id:ID', right_on='post').fillna(0)
posts_df = posts_df.merge(post_comment_counts, how='left', left_on='id', right_on='post').fillna(0)
posts_df['pcNum'] = posts_df['pcNum'].astype(int)

# 按照 pcNum 降序排列 posts 数据
print("按照 pcNum 降序排列帖子数据...")
posts_df = posts_df.sort_values(by='pcNum', ascending=False)


# 保存更新后的 posts_test_large.csv
posts_updated_file = base_path + 'sorted_post_sta.csv'
posts_df.to_csv(posts_updated_file, index=False)
print(f"更新后的 posts 文件保存到 {posts_updated_file}")

# 计算每个 user 发布的评论数
print("计算每个 user 发布的评论数...")
user_comment_counts = comments_df.groupby('creator').size().reset_index(name='ucNum')
# users_df = users_df.merge(user_comment_counts, how='left', left_on='id:ID', right_on='creator').fillna(0)
users_df = users_df.merge(user_comment_counts, how='left', left_on='id', right_on='creator').fillna(0)
users_df['ucNum'] = users_df['ucNum'].astype(int)

# 计算每个 user 发布的帖子数
print("计算每个 user 发布的帖子数...")
user_post_counts = posts_df.groupby('creator').size().reset_index(name='upNum')
# users_df = users_df.merge(user_post_counts, how='left', left_on='id:ID', right_on='creator').fillna(0)
users_df = users_df.merge(user_post_counts, how='left', left_on='id', right_on='creator').fillna(0)
users_df['upNum'] = users_df['upNum'].astype(int)

# 按照 ucNum 降序排列
print("按照 ucNum 降序排列用户数据...")
users_df = users_df.sort_values(by='ucNum', ascending=False)

# 保存更新后的 users_test.csv
users_updated_file = base_path + 'sorted_user_sta.csv'
users_df.to_csv(users_updated_file, index=False)
print(f"更新后的 users 文件保存到 {users_updated_file}")


### 5.读取posts_test_large.csv,users_test.csv,然后文件夹/home/wangshuo/resource/DataSets/parler/csv_data/pc/comments下面有31个comments.csv文件，
comments文件中元组通过外键post和posts文件的id相连接 ，
扫描这31个comments.csv文件，
comments文件中元组通过外键post和posts文件的id相连接 ，
posts 和comments文件中元组通过creator与users的id连接，
对于每个comment元组，如果其外键post可以在posts_test_large找到，且其外键creator可以在users_test中找到，将这个元组保存下来，
最后这些保存下来的元组集合写到文件comments2_test_large.csv文件中

In [ ]:
import os
import pandas as pd
from tqdm import tqdm

# 设置文件路径
posts_test_large_file = '/home/wangshuo/resource/DataSets/parler/csv_data/sub_data/posts_test_large.csv'
users_test_file = '/home/wangshuo/resource/DataSets/parler/csv_data/sub_data/users_test.csv'
comments_dir = '/home/wangshuo/resource/DataSets/parler/csv_data/pc/comments'
output_file = '/home/wangshuo/resource/DataSets/parler/csv_data/sub_data/comments2_test_large.csv'

# 读取posts和users文件
print("读取 posts_test_large.csv 文件...")
posts_test_large_df = pd.read_csv(posts_test_large_file)

print("读取 users_test.csv 文件...")
users_test_df = pd.read_csv(users_test_file)

# 获取有效的post和creator集合
valid_posts = set(posts_test_large_df['id'])
valid_users = set(users_test_df['id'])

# 初始化结果数据框
filtered_comments_list = []

# 扫描comments目录
comment_files = [f for f in os.listdir(comments_dir) if f.endswith('.csv')]

print("开始处理 comments 文件...")
for comment_file in tqdm(comment_files, desc="扫描 comments 文件"):
    # 读取每个评论文件
    comment_file_path = os.path.join(comments_dir, comment_file)
    comments_df = pd.read_csv(comment_file_path)

    # 筛选出post和creator均有效的评论
    valid_comments = comments_df[(comments_df['post'].isin(valid_posts)) & (comments_df['creator'].isin(valid_users))]

    # 添加到结果列表
    filtered_comments_list.append(valid_comments)

# 合并所有过滤后的评论数据
filtered_comments_df = pd.concat(filtered_comments_list, ignore_index=True)

# 保存结果到文件
print("保存过滤后的评论数据到 comments2_test_large.csv...")
filtered_comments_df.to_csv(output_file, index=False)

print("处理完成，保存的评论数据数量：", len(filtered_comments_df))

### 6.读取posts_test_large.csv,comments2_test_large.csv文件，comments文件中元组通过外键post和posts文件的id相连接 ，
对于每个post如果其id在comments2_test_large文件中的post属性能够找到，就保存这个post，
最后这些保存下来的元组集合写到文件posts2_test_large.csv文件中


In [ ]:
import pandas as pd

# 文件路径
posts_test_large_file = '/home/wangshuo/resource/DataSets/parler/csv_data/sub_data/posts_test_large.csv'
comments2_test_large_file = '/home/wangshuo/resource/DataSets/parler/csv_data/sub_data/comments2_test_large.csv'
output_file = '/home/wangshuo/resource/DataSets/parler/csv_data/sub_data/posts2_test_large.csv'

# 读取文件
print("读取 posts_test_large.csv 文件...")
posts_test_large_df = pd.read_csv(posts_test_large_file)

print("读取 comments2_test_large.csv 文件...")
comments2_test_large_df = pd.read_csv(comments2_test_large_file)

# 获取 comments2_test_large 中的有效 post 集合
valid_posts = set(comments2_test_large_df['post'])

# 筛选出有效的 posts 元组
filtered_posts_df = posts_test_large_df[posts_test_large_df['id'].isin(valid_posts)]

# 保存结果到文件
print("保存过滤后的 posts 数据到 posts2_test_large.csv...")
filtered_posts_df.to_csv(output_file, index=False)

print("处理完成，保存的 posts 数据数量：", len(filtered_posts_df))


####7. 检查每个post是否对应一个user，每个comment是否对应一个post，每个comment是否对应一个user，如果不是则从对应文件删除该数据

7.1 删除没有对应post或user的comment，将有效的comment保存到comment_valid.csv中

In [ ]:
#!/usr/bin/env python3
import pandas as pd
from pathlib import Path
import sys

# ========== 配置区（改成你的路径） ==========
datadir = Path("/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5")
user_file = datadir / "user.csv"
post_file = datadir / "post_valid.csv"
comment_file = datadir / "comment.csv"

# 输出文件
backup_comment_file = datadir / "comment_backup_before_filter.csv"
valid_comment_file = datadir / "comment_valid.csv"
removed_comment_file = datadir / "comment_removed_invalid_refs.csv"
# ============================================

def find_id_col(df):
    """优先返回包含 ':ID' 的列，其次以 'id' 开头的列，否则返回第一列名。"""
    for c in df.columns:
        if ":ID" in c:
            return c
    for c in df.columns:
        if c.lower().startswith("id"):
            return c
    return df.columns[0]

def find_creator_col(df):
    """识别 creator 列名（优先 creator:ID，再 creator，再包含 creator 的列）。"""
    for c in df.columns:
        if c.lower().startswith("creator:"):
            return c
    for c in df.columns:
        if c.lower() == "creator" or c.lower().endswith("creator"):
            return c
    for c in df.columns:
        if "creator" in c.lower():
            return c
    return None

def find_post_col(df):
    """识别 post 指向列（优先 post:ID，再 post，再包含 post 的列）。"""
    for c in df.columns:
        if c.lower().startswith("post:"):
            return c
    for c in df.columns:
        if c.lower() == "post" or c.lower().endswith("post"):
            return c
    for c in df.columns:
        if "post" in c.lower():
            return c
    return None

def load_csv(path):
    if not path.exists():
        print(f"ERROR: 找不到文件: {path}")
        sys.exit(1)
    return pd.read_csv(path, dtype=str, keep_default_na=False)

def pct(n, total):
    return f"{n} ({n/total*100:.2f}%)" if total>0 else f"{n} (N/A)"

def main():
    # 读取
    user_df = load_csv(user_file)
    post_df = load_csv(post_file)
    comment_df = load_csv(comment_file)

    # 识别列
    user_id_col = find_id_col(user_df)
    post_id_col = find_id_col(post_df)
    comment_id_col = find_id_col(comment_df)
    comment_creator_col = find_creator_col(comment_df)
    comment_post_col = find_post_col(comment_df)

    print("Detected columns:")
    print(" user id:", user_id_col)
    print(" post id:", post_id_col)
    print(" comment id:", comment_id_col)
    print(" comment.creator:", comment_creator_col)
    print(" comment.post:", comment_post_col)
    print()

    if comment_creator_col is None:
        print("ERROR: 未识别 comment 的 creator 列，请确认表头包含 'creator' 或 'creator:ID' 等。")
        sys.exit(1)
    if comment_post_col is None:
        print("ERROR: 未识别 comment 的 post 列，请确认表头包含 'post' 或 'post:ID' 等。")
        sys.exit(1)

    # 规范化 id 值（strip 空白）
    user_ids = set(user_df[user_id_col].astype(str).str.strip())
    post_ids = set(post_df[post_id_col].astype(str).str.strip())

    # 处理 comment 字段
    comment_df[comment_creator_col] = comment_df[comment_creator_col].astype(str).str.strip()
    comment_df[comment_post_col] = comment_df[comment_post_col].astype(str).str.strip()

    # 计算 mask：creator 在 user && post 在 post
    mask_creator_ok = comment_df[comment_creator_col].isin(user_ids)
    mask_post_ok = comment_df[comment_post_col].isin(post_ids)
    mask_both_ok = mask_creator_ok & mask_post_ok

    total_comments = len(comment_df)
    valid_comments = comment_df[mask_both_ok].copy()
    removed_comments = comment_df[~mask_both_ok].copy()

    # 备份原 comment
    # comment_df.to_csv(backup_comment_file, index=False, encoding="utf-8")
    # 保存结果
    valid_comments.to_csv(valid_comment_file, index=False, encoding="utf-8")
    removed_comments.to_csv(removed_comment_file, index=False, encoding="utf-8")

    # 输出统计
    print("Total comments:", total_comments)
    print("Valid comments (creator in user AND post in post):", pct(len(valid_comments), total_comments))
    print("Removed comments (missing user or post):", pct(len(removed_comments), total_comments))
    print()
    # 更细的拆分统计
    only_creator_missing = (~mask_creator_ok) & mask_post_ok
    only_post_missing = mask_creator_ok & (~mask_post_ok)
    both_missing = (~mask_creator_ok) & (~mask_post_ok)
    print(" Breakdown (mutually exclusive):")
    print("  only creator missing, post exists:", pct(only_creator_missing.sum(), total_comments))
    print("  only post missing, creator exists:", pct(only_post_missing.sum(), total_comments))
    print("  both missing:", pct(both_missing.sum(), total_comments))
    print()

    # 打印少量示例供检查
    SAMPLE = 10
    if len(removed_comments) > 0:
        print("Sample removed comment ids (up to %d):" % SAMPLE)
        print(removed_comments[comment_id_col].head(SAMPLE).to_list())

    print("\nFiles written:")
    # print(" backup original comment ->", backup_comment_file)
    print(" valid comments ->", valid_comment_file)
    print(" removed comments ->", removed_comment_file)
    print("\nDone.")

if __name__ == "__main__":
    main()


7.2 删除没有对应user的post，将有效的post保存到post_valid.csv中

In [ ]:
#!/usr/bin/env python3
import pandas as pd
from pathlib import Path
import sys

# ========== 配置区（改成你的路径） ==========
datadir = Path("/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5")
user_file = datadir / "user.csv"
post_file = datadir / "post.csv"

# 输出（不会覆盖原始文件）
post_valid_file = datadir / "post_valid.csv"
post_removed_file = datadir / "post_removed_no_user.csv"
# ============================================

def find_id_col(df):
    for c in df.columns:
        if ":ID" in c:
            return c
    for c in df.columns:
        if c.lower().startswith("id"):
            return c
    return df.columns[0]

def find_creator_col(df):
    for c in df.columns:
        if c.lower().startswith("creator:"):
            return c
    for c in df.columns:
        if c.lower() == "creator" or c.lower().endswith("creator"):
            return c
    for c in df.columns:
        if "creator" in c.lower():
            return c
    return None

def load_csv(path):
    if not path.exists():
        print(f"ERROR: 找不到文件: {path}")
        sys.exit(1)
    return pd.read_csv(path, dtype=str, keep_default_na=False)

def main():
    user_df = load_csv(user_file)
    post_df = load_csv(post_file)

    user_id_col = find_id_col(user_df)
    post_id_col = find_id_col(post_df)
    post_creator_col = find_creator_col(post_df)

    print("Detected columns:")
    print(" user id column:", user_id_col)
    print(" post id column:", post_id_col)
    print(" post.creator column:", post_creator_col)
    print()

    if post_creator_col is None:
        print("ERROR: 未识别 post 表中的 creator 列（如 'creator' 或 'creator:ID'），请检查 post.csv 的表头。")
        sys.exit(1)

    # 规范化并构造集合
    user_ids = set(user_df[user_id_col].astype(str).str.strip())
    post_df[post_creator_col] = post_df[post_creator_col].astype(str).str.strip()
    post_df[post_id_col] = post_df[post_id_col].astype(str).str.strip()

    total_posts = len(post_df)
    mask_creator_exists = post_df[post_creator_col].isin(user_ids)
    kept_posts_df = post_df[mask_creator_exists].copy()
    removed_posts_df = post_df[~mask_creator_exists].copy()

    # 写出结果（不修改原文件）
    kept_posts_df.to_csv(post_valid_file, index=False, encoding="utf-8")
    removed_posts_df.to_csv(post_removed_file, index=False, encoding="utf-8")

    # 输出统计
    def pct(n, tot):
        return f"{n} ({n/tot*100:.2f}%)" if tot>0 else f"{n} (N/A)"

    print("Total posts:", total_posts)
    print("Kept posts (creator exists in user):", pct(len(kept_posts_df), total_posts))
    print("Removed posts (creator missing):", pct(len(removed_posts_df), total_posts))
    if len(removed_posts_df) > 0:
        print("\nSample removed post ids (up to 10):")
        print(removed_posts_df[post_id_col].head(10).tolist())

    print("\nFiles generated (original files untouched):")
    print(" ", post_valid_file)
    print(" ", post_removed_file)

if __name__ == "__main__":
    main()


7.3 删除没有发表post和comment的user，将有效的user保存到user_valid.csv中

In [ ]:
#!/usr/bin/env python3
import pandas as pd
from pathlib import Path
import sys

# ========== 配置区（改成你的路径） ==========
datadir = Path("/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5")
user_file = datadir / "user.csv"
post_file = datadir / "post_valid.csv"
comment_file = datadir / "comment_valid.csv"

# 输出文件（不会覆盖原始 user.csv）
user_valid_file = datadir / "user_valid.csv"
user_removed_file = datadir / "user_removed_no_activity.csv"
# ============================================

def find_id_col(df):
    for c in df.columns:
        if ":ID" in c:
            return c
    for c in df.columns:
        if c.lower().startswith("id"):
            return c
    return df.columns[0]

def find_creator_col(df):
    # 优先 creator:ID，其次 creator、包含 creator 的列
    for c in df.columns:
        if c.lower().startswith("creator:"):
            return c
    for c in df.columns:
        if c.lower() == "creator" or c.lower().endswith("creator"):
            return c
    for c in df.columns:
        if "creator" in c.lower():
            return c
    return None

def load_csv(path):
    if not path.exists():
        print(f"ERROR: 找不到文件: {path}")
        sys.exit(1)
    return pd.read_csv(path, dtype=str, keep_default_na=False)

def main():
    # 读取数据
    user_df = load_csv(user_file)
    post_df = load_csv(post_file)
    comment_df = load_csv(comment_file)

    # 识别列
    user_id_col = find_id_col(user_df)
    post_creator_col = find_creator_col(post_df)
    comment_creator_col = find_creator_col(comment_df)

    print("Detected columns:")
    print(" user id:", user_id_col)
    print(" post.creator:", post_creator_col)
    print(" comment.creator:", comment_creator_col)
    print()

    # 标准化并构造活跃用户集合（来自 post 或 comment 的 creator）
    active_user_ids = set()

    if post_creator_col:
        post_creators = post_df[post_creator_col].astype(str).str.strip()
        active_user_ids.update(post_creators[post_creators != ""].tolist())
    else:
        print("WARN: 未识别 post 表中的 creator 列，post 贡献的活跃用户将被忽略。")

    if comment_creator_col:
        comment_creators = comment_df[comment_creator_col].astype(str).str.strip()
        active_user_ids.update(comment_creators[comment_creators != ""].tolist())
    else:
        print("WARN: 未识别 comment 表中的 creator 列，comment 贡献的活跃用户将被忽略。")

    # 去除空字符串
    active_user_ids = {uid for uid in active_user_ids if uid != ""}

    total_users = len(user_df)
    print("Total users (original):", total_users)
    print("Active user ids detected (unique):", len(active_user_ids))

    # 标准化 user id 列并筛选
    user_df[user_id_col] = user_df[user_id_col].astype(str).str.strip()
    mask_active = user_df[user_id_col].isin(active_user_ids)
    user_valid_df = user_df[mask_active].copy()
    user_removed_df = user_df[~mask_active].copy()

    # 写出结果（不覆盖原始 user.csv）
    user_valid_df.to_csv(user_valid_file, index=False, encoding="utf-8")
    user_removed_df.to_csv(user_removed_file, index=False, encoding="utf-8")

    # 输出统计和示例
    def pct(n, tot):
        return f"{n} ({n/tot*100:.2f}%)" if tot>0 else f"{n} (N/A)"

    print("\nResult:")
    print(" Valid users ->", user_valid_file.name, ":", pct(len(user_valid_df), total_users))
    print(" Removed users (no post & no comment) ->", user_removed_file.name, ":", pct(len(user_removed_df), total_users))
    if len(user_removed_df) > 0:
        SAMPLE = 10
        print(" Sample removed user ids (up to {}):".format(SAMPLE))
        print(user_removed_df[user_id_col].head(SAMPLE).tolist())

    print("\nDone. Original files untouched.")

if __name__ == "__main__":
    main()


7.4 再次检查统计，是否还有comment缺少user和post，post缺少user，user没有发表任何post和comment

In [8]:
#!/usr/bin/env python3
import pandas as pd
from pathlib import Path
import sys

# ========== 配置区 ==========
datadir = Path("/home/wangshuo/resource/datasets/parler_data/30W_valid_user/10-10-5")
user_file = datadir / "user_valid.csv"
post_file = datadir / "post_valid.csv"
comment_file = datadir / "comment_valid.csv"
# ============================

def find_id_col(df):
    for c in df.columns:
        if ":ID" in c:
            return c
    for c in df.columns:
        if c.lower().startswith("id"):
            return c
    return df.columns[0]

def find_creator_col(df):
    for c in df.columns:
        if c.lower().startswith("creator:"):
            return c
    for c in df.columns:
        if c.lower() == "creator" or c.lower().endswith("creator"):
            return c
    for c in df.columns:
        if "creator" in c.lower():
            return c
    return None

def find_post_col(df):
    for c in df.columns:
        if c.lower().startswith("post:"):
            return c
    for c in df.columns:
        if c.lower() == "post" or c.lower().endswith("post"):
            return c
    for c in df.columns:
        if "post" in c.lower():
            return c
    return None

def load_csv(path):
    if not path.exists():
        print(f"ERROR: 找不到文件: {path}")
        sys.exit(1)
    return pd.read_csv(path, dtype=str, keep_default_na=False)

def pct(n, total):
    return f"{n} ({n/total*100:.2f}%)" if total > 0 else f"{n} (N/A)"

def main():
    # 读取
    user_df = load_csv(user_file)
    post_df = load_csv(post_file)
    comment_df = load_csv(comment_file)

    # 识别列
    user_id_col = find_id_col(user_df)
    post_id_col = find_id_col(post_df)
    post_creator_col = find_creator_col(post_df)
    comment_id_col = find_id_col(comment_df)
    comment_creator_col = find_creator_col(comment_df)
    comment_post_col = find_post_col(comment_df)

    print("Detected columns:")
    print(" user id:", user_id_col)
    print(" post id:", post_id_col, ", post.creator:", post_creator_col)
    print(" comment id:", comment_id_col, ", comment.creator:", comment_creator_col, ", comment.post:", comment_post_col)
    print()

    # ========== 规范化 ID ==========
    user_df[user_id_col] = user_df[user_id_col].astype(str).str.strip()
    post_df[post_id_col] = post_df[post_id_col].astype(str).str.strip()
    comment_df[comment_id_col] = comment_df[comment_id_col].astype(str).str.strip()

    user_ids = set(user_df[user_id_col])
    post_ids = set(post_df[post_id_col])

    # 1. Comment 检查
    total_comments = len(comment_df)
    mask_creator_ok = comment_df[comment_creator_col].isin(user_ids) if comment_creator_col else pd.Series([False]*total_comments)
    mask_post_ok = comment_df[comment_post_col].isin(post_ids) if comment_post_col else pd.Series([False]*total_comments)

    only_creator_missing = (~mask_creator_ok) & mask_post_ok
    only_post_missing = mask_creator_ok & (~mask_post_ok)
    both_missing = (~mask_creator_ok) & (~mask_post_ok)

    print("=== Comment consistency ===")
    print(" Total comments:", total_comments)
    print("  valid:", pct((mask_creator_ok & mask_post_ok).sum(), total_comments))
    print("  only creator missing:", pct(only_creator_missing.sum(), total_comments))
    print("  only post missing:", pct(only_post_missing.sum(), total_comments))
    print("  both missing:", pct(both_missing.sum(), total_comments))
    print()

    # 2. Post 检查
    total_posts = len(post_df)
    mask_post_creator_ok = post_df[post_creator_col].isin(user_ids) if post_creator_col else pd.Series([False]*total_posts)

    print("=== Post consistency ===")
    print(" Total posts:", total_posts)
    print("  creator exists in user:", pct(mask_post_creator_ok.sum(), total_posts))
    print("  creator missing:", pct((~mask_post_creator_ok).sum(), total_posts))
    print()

    # 3. User 活跃度检查
    active_user_ids = set()
    if post_creator_col:
        active_user_ids.update(post_df[post_creator_col].tolist())
    if comment_creator_col:
        active_user_ids.update(comment_df[comment_creator_col].tolist())
    active_user_ids = {u.strip() for u in active_user_ids if u.strip() != ""}

    total_users = len(user_df)
    mask_active = user_df[user_id_col].isin(active_user_ids)

    print("=== User activity ===")
    print(" Total users:", total_users)
    print("  active (has post or comment):", pct(mask_active.sum(), total_users))
    print("  inactive (no post & no comment):", pct((~mask_active).sum(), total_users))
    print()

    print("Check finished.")

if __name__ == "__main__":
    main()


Detected columns:
 user id: id:ID
 post id: id:ID , post.creator: creator
 comment id: id:ID , comment.creator: creator , comment.post: post
=== Comment consistency ===
 Total comments: 110264
  valid: 110264 (100.00%)
  only creator missing: 0 (0.00%)
  only post missing: 0 (0.00%)
  both missing: 0 (0.00%)

=== Post consistency ===
 Total posts: 24082
  creator exists in user: 24082 (100.00%)
  creator missing: 0 (0.00%)

=== User activity ===
 Total users: 41342
  active (has post or comment): 41342 (100.00%)
  inactive (no post & no comment): 0 (0.00%)

Check finished.
